# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. This dataset details protein analysis using mass spectrometry on extracellular vesicles from human mast cells, annotated with the Croissant metadata schema.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata
dataset = mlc.Dataset(croissant_url)
# Get and print summary description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Let's review the available record sets and their fields, referencing each by their Croissant `@id` field.

We'll list record sets, their `@id`, and, for each record set, all available fields and columns with their Croissant `@id` and name for orientation.

In [ ]:
# List all record sets, their ids, and their fields/columns
def print_croissant_overview(ds):
    print("--- Record Sets Available ---")
    if not ds.metadata.record_sets:
        print("No record sets defined in this Croissant schema.")
        return []
    ids = []
    for rs in ds.metadata.record_sets:
        print(f"Record Set {rs.metadata['@id'] if '@id' in rs.metadata else '(unknown)'}: {rs.name}")
        ids.append(rs.metadata.get('@id', None))
        if rs.fields:
            print("  Fields:")
            for field in rs.fields:
                print(f"    - {field.metadata.get('@id','?')}: {field.name} (type={field.data_type})")
        if hasattr(rs, 'columns') and rs.columns:
            print("  Columns:")
            for col in rs.columns:
                print(f"    - {col.metadata.get('@id','?')}: {col.name} (type={col.data_type})")
        print()
    return ids

record_set_ids = print_croissant_overview(dataset)

## 3. Data Extraction

We extract data from one or more record sets, referencing each by their Croissant `@id`.

**Example:** If a record set is listed with `@id` `cr:recordSet_1`, use that as its identifier below.

In [ ]:
# If there are record sets, load them into pandas DataFrames (referenced by @id)
dataframes = {}

if not record_set_ids:
    print("No record sets available in this Croissant schema!")
else:
    for record_set_id in record_set_ids:
        print(f"Loading records from record set {record_set_id} ...")
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print(f"No records found for record set {record_set_id}.")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        display(df.head(2))

## 4. Exploratory Data Analysis (EDA)

Let’s process and analyze the loaded data. We'll filter, normalize, and group by fields, referencing all field and column names by their `@id` from the Croissant metadata.

- **Choose a numeric field (via Croissant `@id`) for filtering/normalization.**
- **Choose a grouping field (via Croissant `@id`) if appropriate.**


In [ ]:
# ----- Customize these according to the printed overview above -----
# For demonstration, we'll auto-select the first DataFrame with at least one numeric column.
import numpy as np

selected_record_set_id = None
numeric_field_id = None
group_field_id = None

# Try to auto-select a numeric column for demonstration
for rid, df in dataframes.items():
    for col in df.columns:
        # Heuristic: try to detect numbers
        if np.issubdtype(df[col].dtype, np.number):
            selected_record_set_id = rid
            numeric_field_id = col
            print(f"Using record set {rid} and numeric field {col} for demonstration.")
            break
    if selected_record_set_id:
        break

if selected_record_set_id is None or numeric_field_id is None:
    print("No numeric field found for demo.")
else:
    # (Optional) try to pick group field
    for col in dataframes[selected_record_set_id].columns:
        if col != numeric_field_id and dataframes[selected_record_set_id][col].nunique() < len(dataframes[selected_record_set_id])//4:
            group_field_id = col
            break

    df = dataframes[selected_record_set_id]
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'iufc' else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records in '{selected_record_set_id}' with {numeric_field_id} > {threshold:.2f}, n={len(filtered_df)}")
    # Normalize
    filtered_df = filtered_df.copy()
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by group_field_id if available
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id} (mean):")
        print(grouped_df[[numeric_field_id, norm_col]].head())

## 5. Visualization

Here we visualize the distribution of our chosen numeric field and the normalized values for the filtered data subset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20, color='skyblue')
    plt.title(f"Histogram of {numeric_field_id} (> mean) in {selected_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, bins=20, color='orange')
    plt.title(f"Normalized Histogram of {numeric_field_id} (> mean)")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.show()
    if group_field_id:
        # Boxplot
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} distribution across {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- This notebook demonstrated end-to-end data access, processing, and visualization of a mass spectrometry protein dataset using `mlcroissant` and Croissant metadata.
- All data, fields, and schema elements were referenced by their Croissant `@id` according to best practices.
- You are encouraged to adjust the EDA and visualizations for your analytical goals or to select other record sets or fields listed above.